# Raman Mapping Backend and Provenance Demo

**v0.4.8 — canonical mapping backend-behaviour and provenance example**

This notebook demonstrates:
1. RamanSPy backend propagation through Raman **mapping** preprocessing
2. Provenance fields recorded in exported metadata
3. How `native` and `auto` backends produce equivalent results

## Backend support summary for mapping

| Backend requested | Condition | Backend resolved |
|-------------------|-----------|------------------|
| `"native"` | always | `"native"` |
| `"auto"` | RamanSPy installed + supported pipeline | `"ramanspy"` |
| `"auto"` | RamanSPy not installed or unsupported pipeline | `"native"` (fallback) |
| `"ramanspy"` | RamanSPy not installed or unsupported pipeline | raises `NotImplementedError` |

In [1]:
import numpy as np
import matplotlib.pyplot as plt

from ramanpl import Mapping
from ramanpl.preprocessing import Pipeline, CropByRange, SmoothSavGol, BaselineSubtract

## 1. Define a fully translatable preprocessing pipeline

All steps are in the supported set, so `auto` will resolve to RamanSPy when available.

In [2]:
custom_peaks = {
    'E2g': ([348, 0.5, 0], [360, 10, 70]),
    'A1g': ([418, 0.5, 0], [424, 10, 70]),
    '2LA(M)':([340, 0.5, 0], [350, 10,  70]),
}
data_range = (300, 600)
step_size = 0.5

PIPELINE_STEPS = [
    CropByRange((330, 440)),
    SmoothSavGol(window_length=11, polyorder=9),
    BaselineSubtract({"method": "arpls", "lam": 1e4, "niter": 250, "tol": 1e-7}),
]

## 2. Run mapping with `backend="native"`

In [ ]:
pipe_native = Pipeline(steps=PIPELINE_STEPS, backend="native")

raman_map_native = Mapping.RamanMapping(
    filename='Mapping Raman Sample.wdf',
    custom_peaks=custom_peaks,
    data_range=data_range,
    normalize=False,
    smoothing=False,
    background_remove=True,
    preprocessing=pipe_native,
    step_size=step_size,
)
_ = raman_map_native.fit_spectra(warm_start=True, seed_coord=(10, 10))
print("Native fit complete")

## 3. Run mapping with `backend="auto"`

With RamanSPy installed and a supported pipeline, `auto` resolves to `ramanspy`.

In [ ]:
pipe_auto = Pipeline(steps=PIPELINE_STEPS, backend="auto")

raman_map_auto = Mapping.RamanMapping(
    filename='Mapping Raman Sample.wdf',
    custom_peaks=custom_peaks,
    data_range=data_range,
    normalize=False,
    smoothing=False,
    background_remove=True,
    preprocessing=pipe_auto,
    step_size=step_size,
)
_ = raman_map_auto.fit_spectra(warm_start=True, seed_coord=(10, 10))
print("Auto fit complete")

Successful fits: 1533 / 1533
Auto fit complete


## 4. Show backend propagation in export metadata

The exported TXT file header records `preprocessing_backend_requested` and
`preprocessing_backend_resolved` for reproducibility. When a fallback occurred,
`preprocessing_backend_fallback_reason` is also present.

In [10]:
import os, io

# Export to a buffer and show the provenance lines from the header
native_path = "outputs/raman_backend_demo_native.txt"
auto_path   = "outputs/raman_backend_demo_auto.txt"

os.makedirs("outputs", exist_ok=True)

raman_map_native.export_fit_map(native_path, coord_mode="pixel")
raman_map_auto.export_fit_map(auto_path,   coord_mode="pixel")

def show_provenance(path):
    print(f"\n--- {path} ---")
    with open(path) as f:
        for line in f:
            if "preprocessing_backend" in line:
                print(line, end="")

show_provenance(native_path)
show_provenance(auto_path)


--- outputs/raman_backend_demo_native.txt ---
# preprocessing_backend_requested: "native"
# preprocessing_backend_resolved: "native"

--- outputs/raman_backend_demo_auto.txt ---
# preprocessing_backend_requested: "auto"
# preprocessing_backend_resolved: "ramanspy"


## 5. Representative intensity map (auto backend result)

In [ ]:
raman_map_auto.plot_heatmap(peak_name="E2g", data_type='intensity')